In [3]:
base_graph = { # Нет цикла/пути
    0: [1, 2],
    1: [0, 3, 4],
    2: [0],
    3: [1],
    4: [1]
}

euler_cycle_graph = { # Есть цикл
    0: [1, 2],
    1: [0, 2],
    2: [0, 1, 3, 4],
    3: [2, 4],
    4: [2, 3]
}

euler_path_graph = { # Есть путь
    0: [1, 2, 3],
    1: [0, 2, 3],
    2: [0, 1],
    3: [0, 1]
}

little_bit_big_graph_1_comp = {
    0: [1, 2, 3],
    1: [0, 4, 5],
    2: [0, 5, 6],
    3: [0, 7],
    4: [1, 8],
    5: [1, 2, 8, 9],
    6: [2, 9],
    7: [3, 9],
    8: [4, 5],
    9: [5, 6, 7]
}

little_bit_big_graph_2_comp = {
    0: [1, 2, 3],
    1: [0, 4, 5],
    2: [0, 5, 6],
    3: [0, 7],
    4: [1, 8],
    5: [1, 2, 8],
    6: [2],
    7: [3],
    8: [4, 5],
    9: []
}

disconnected_graph = {
    0: [1, 2],
    1: [0, 2],
    2: [0, 1],
    3: [4, 5],
    4: [3, 5],
    5: [3, 4],
    6: [7],
    7: [6, 8],
    8: [7],
    9: []
}

graph_with_cycle = {
    1: [2, 4],
    2: [1, 3],
    3: [2, 4],
    4: [3, 1]
}

graph_with_path_only = {
    1: [2, 4, 3],
    2: [1, 3],
    3: [2, 4, 1],
    4: [3, 1]
}

complex_eulerian_cycle = {
    0: [1, 2, 3, 4],
    1: [0, 2, 5, 6],
    2: [0, 1, 3, 7],
    3: [0, 2, 4, 5],
    4: [0, 3, 6, 7],
    5: [1, 3, 6, 7],
    6: [1, 4, 5, 7],
    7: [2, 4, 5, 6]
}

complex_eulerian_path = {
    0: [1, 2, 3, 4],
    1: [0, 2, 5, 6],
    2: [0, 1, 3, 7],
    3: [0, 2, 4],
    4: [0, 3, 6, 7],
    5: [1, 6, 7],
    6: [1, 4, 5, 7],
    7: [2, 4, 5, 6]
}

In [20]:
import copy
import urllib

class Fluri:
    def __init__(self, graph = None):
        self.graph = graph
        
    def _isConnected(self):
        if not self.graph: 
            return True
        
        start_node = next(iter(self.graph))
        visited_nodes = set()
        nodes_needed_visited = [start_node] 
        
        while nodes_needed_visited:
            node = nodes_needed_visited.pop()
            
            if node not in visited_nodes:
                visited_nodes.add(node)
                for neighbor in self.graph[node]:
                    if neighbor not in visited_nodes:
                        nodes_needed_visited.append(neighbor)
                        
        return len(visited_nodes) == len(self.graph)

    def _does_exist_path(self):
        if self._isConnected(): # Проверяем является ли граф связным
            k = 0
            for i in self.graph:
                if len(self.graph[i]) % 2 != 0: # Условие, что все вершины имеют четную степерь
                    k+=1
                if k > 2: # Также выполняется условие, если только у ДВУХ вершин нечетная степень
                    print("В графе нет цикла/пути")
                    return False
            if k == 1:
                print("В графе нет цикла/пути")
                return False

            return True
        else:
            print("В графе нет цикла/пути")
            return False

    def _delete_edge(self, main_node, candidate):
        self.graph[main_node].remove(candidate)
        self.graph[candidate].remove(main_node)

    def _does_bridge(self, main_node, candidate):
        self._delete_edge(main_node, candidate)

        is_bridge = not self._isConnected()

        self.graph[main_node].append(candidate)
        self.graph[candidate].append(main_node)

        return is_bridge
        
    def find(self, original_graph):
        self.graph = copy.deepcopy(original_graph)

        if not self._does_exist_path():
            return False

        start_node = next(iter(self.graph))
        target = 'цикл'
        for i in self.graph:
            if len(self.graph[i])%2==1:
                target = 'путь'
                start_node = i
                break

        main_node = start_node 
        path = [start_node]

        while len(self.graph[main_node]) > 0: 
            neighbor = self.graph[main_node]
            
            if len(neighbor) == 1: 
                # Если ребро единственное — у нас нет выбора, идем по нему
                next_node = neighbor[0] 
                self._delete_edge(main_node, next_node) # Удаляем ребро
                main_node = next_node
                path.append(main_node)
            
            else:
                # Если кандидатов несколько, анализируем их
                chosen_node = None
                for i in neighbor:
                    if not self._does_bridge(main_node, i):
                        chosen_node = i
                        break
                
                if chosen_node == None:
                    chosen_node = neighbor[0]

                self._delete_edge(main_node, chosen_node) # Удаляем ребро
                main_node = chosen_node
                path.append(main_node)
        print(f'В графе есть {target}', path)
        return path

def export_to_dot(graph, graph_name="G"):
    """Экспортирует неориентированный граф в формат DOT."""
    lines = [f"graph {graph_name} {{"]
    seen_edges = set()
    
    for u, neighbors in graph.items():
        if not neighbors:
            lines.append(f"    \"{u}\";")
            
        for v in neighbors:
            edge = tuple(sorted((str(u), str(v))))
            
            if edge not in seen_edges:
                seen_edges.add(edge)
                lines.append(f"    \"{u}\" -- \"{v}\";")
                
    lines.append("}")

    encoded_dot = urllib.parse.quote( "\n".join(lines))

    return f"https://dreampuf.github.io/GraphvizOnline/#{encoded_dot}"

In [26]:
assert Fluri().find(base_graph) == False
assert Fluri().find(little_bit_big_graph_1_comp) == False
assert Fluri().find(little_bit_big_graph_2_comp) == False
assert Fluri().find(disconnected_graph) == False
print()
assert isinstance(Fluri().find(euler_cycle_graph), list)
assert isinstance(Fluri().find(complex_eulerian_cycle), list)
print()
assert isinstance(Fluri().find(complex_eulerian_path), list)
assert isinstance(Fluri().find(euler_path_graph), list)
assert isinstance(Fluri().find(graph_with_path_only), list)

В графе нет цикла/пути
В графе нет цикла/пути
В графе нет цикла/пути
В графе нет цикла/пути

В графе есть цикл [0, 1, 2, 3, 4, 2, 0]
В графе есть цикл [0, 1, 2, 0, 3, 2, 7, 5, 3, 4, 6, 7, 4, 0]

В графе есть путь [3, 0, 1, 2, 0, 4, 6, 5, 7, 4, 3, 2, 7, 6, 1, 5]
В графе есть путь [0, 1, 2, 0, 3, 1]
В графе есть путь [1, 2, 3, 1, 4, 3]


In [27]:
export_to_dot(complex_eulerian_cycle)

'https://dreampuf.github.io/GraphvizOnline/#graph%20G%20%7B%0A%20%20%20%20%220%22%20--%20%221%22%3B%0A%20%20%20%20%220%22%20--%20%222%22%3B%0A%20%20%20%20%220%22%20--%20%223%22%3B%0A%20%20%20%20%220%22%20--%20%224%22%3B%0A%20%20%20%20%221%22%20--%20%222%22%3B%0A%20%20%20%20%221%22%20--%20%225%22%3B%0A%20%20%20%20%221%22%20--%20%226%22%3B%0A%20%20%20%20%222%22%20--%20%223%22%3B%0A%20%20%20%20%222%22%20--%20%227%22%3B%0A%20%20%20%20%223%22%20--%20%224%22%3B%0A%20%20%20%20%223%22%20--%20%225%22%3B%0A%20%20%20%20%224%22%20--%20%226%22%3B%0A%20%20%20%20%224%22%20--%20%227%22%3B%0A%20%20%20%20%225%22%20--%20%226%22%3B%0A%20%20%20%20%225%22%20--%20%227%22%3B%0A%20%20%20%20%226%22%20--%20%227%22%3B%0A%7D'